In [1]:
import sqlite3 as sql
import pandas as pd

In [2]:
conn = sql.connect("ecommerce.db")

In [4]:
sales = pd.read_csv('data/sales.csv')
sales.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

506290

In [5]:
query = """
SELECT *
FROM sales
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [6]:
query = """
SELECT
    COUNT(*) AS Transactions,
    SUM(Quantity) AS Total_Units,
    SUM(Revenue) AS Total_Revenue
FROM sales;
"""

result = pd.read_sql_query(query, conn)

print(result)

   Transactions  Total_Units  Total_Revenue
0        506290      5992597   1.027176e+07


In [7]:
query = """
SELECT
    COUNT(*) AS Transaction_Rows,
    COUNT(DISTINCT Invoice) AS Orders,
    SUM(Quantity) AS Total_Units,
    SUM(Revenue) AS Total_Revenue
FROM sales;
"""

result = pd.read_sql_query(query, conn)

print(result)

   Transaction_Rows  Orders  Total_Units  Total_Revenue
0            506290   22100      5992597   1.027176e+07


In [8]:
query = """
SELECT
    Country,
    SUM(Revenue) AS Revenue
FROM sales
GROUP BY Country
ORDER BY Revenue DESC
LIMIT 10;
"""

country_revenue_sql = pd.read_sql_query(query, conn)

print(country_revenue_sql)

          Country      Revenue
0  United Kingdom  8812311.833
1            EIRE   380909.570
2     Netherlands   268784.350
3         Germany   202025.391
4          France   147103.140
5          Sweden    53501.990
6         Denmark    50906.850
7           Spain    47568.650
8     Switzerland    43921.390
9       Australia    31446.800


In [2]:
import sqlite3 as sql
conn = sql.connect('ecommerce.db')

In [4]:
import pandas as pd
query = """
WITH customer_orders AS (
    SELECT
        "Customer ID",
        COUNT(DISTINCT Invoice) AS Orders
    FROM sales
    WHERE "Customer ID" IS NOT NULL
    GROUP BY "Customer ID"
)

SELECT
    CASE
        WHEN Orders = 1 THEN 'One-time'
        ELSE 'Repeat'
    END AS Customer_Type,
    COUNT(*) AS Customers
FROM customer_orders
GROUP BY Customer_Type
ORDER BY Customers DESC;
"""

customer_type_sql = pd.read_sql_query(query, conn)

print(customer_type_sql)

  Customer_Type  Customers
0        Repeat       2893
1      One-time       1421


In [5]:
query = """
WITH customer_revenue AS (
    SELECT
        "Customer ID",
        SUM(Revenue) AS Revenue
    FROM sales
    WHERE "Customer ID" IS NOT NULL
    GROUP BY "Customer ID"
)

SELECT
    "Customer ID",
    Revenue,
    RANK() OVER (ORDER BY Revenue DESC) AS Revenue_Rank
FROM customer_revenue
ORDER BY Revenue DESC
LIMIT 10;
"""

top_customers_sql = pd.read_sql_query(query, conn)

print(top_customers_sql)

   Customer ID    Revenue  Revenue_Rank
0      18102.0  349164.35             1
1      14646.0  248396.50             2
2      14156.0  196549.74             3
3      14911.0  152121.22             4
4      13694.0  131443.19             5
5      17511.0   84541.17             6
6      15061.0   83284.38             7
7      16684.0   80489.21             8
8      16754.0   65500.07             9
9      17949.0   60117.60            10
